# Predizione del vincitore di una partita di tennis con un Decision Tree (criterio Gini)

## Obiettivo

Il problema e' una **classificazione binaria**:

- classe `1`: vince il giocatore A;
- classe `0`: vince il giocatore B.

L'obiettivo e' addestrare un `DecisionTreeClassifier` di scikit-learn con
criterio Gini, confrontare le prestazioni con e senza le feature derivate
dalle quote di mercato, e verificare se il modello generalizza sulle
partite disputate dopo quelle usate per l'addestramento.

## Perche' un Decision Tree con criterio Gini?

Un singolo albero decisionale e' facilmente interpretabile: la struttura
dell'albero e le regole di split possono essere ispezionate direttamente,
a differenza di un ensemble come CatBoost. Il criterio Gini misura
l'impurita' di un nodo e guida la scelta dello split che la riduce di piu'.

Per contenere l'overfitting tipico di un albero singolo, gli iperparametri
(profondita' massima, dimensione minima delle foglie, dimensione minima di
split, cost-complexity pruning `ccp_alpha`) vengono selezionati tramite
grid search, con fold di cross-validation costruiti su **date uniche**
(non sull'indice di riga) cosi' le due osservazioni simmetriche generate da
ogni match restano sempre nello stesso fold.

## Configurazione

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

PROJECT_ROOT = Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT / "scripts" / "models"))

from training_utils import (  # noqa: E402
    chronological_holdout_split,
    load_feature_dataset,
    make_date_based_folds,
)

FEATURES_PATH = PROJECT_ROOT / "data/interim/tennis_matches_features.data"
METADATA_PATH = PROJECT_ROOT / "data/interim/tennis_matches_features_metadata.json"

HOLDOUT_FRACTION = 0.15
CV_FOLDS = 5
ODDS_FEATURES = {
    "has_any_market_odds",
    "winner_implied_prob_avg",
    "loser_implied_prob_avg",
    "implied_prob_diff_avg",
}

RANDOM_SEED = 42

## 1. Caricamento del dataset

In [ ]:
dataset_with_odds = load_feature_dataset(FEATURES_PATH, METADATA_PATH)
dataset_without_odds = load_feature_dataset(
    FEATURES_PATH, METADATA_PATH, excluded_features=ODDS_FEATURES
)

print("Righe:", len(dataset_with_odds.features))
print("Feature numeriche (con quote):", len(dataset_with_odds.numeric_features))
print("Feature categoriche (con quote):", len(dataset_with_odds.categorical_features))
print("Feature numeriche (senza quote):", len(dataset_without_odds.numeric_features))

preview = dataset_with_odds.features.copy()
preview["date"] = dataset_with_odds.dates
preview["player_a_win"] = dataset_with_odds.target
display(preview.head())

Il dataset contiene due righe per ogni partita reale, una per ciascun
orientamento dei giocatori (`player_a_win` bilanciato). Sono usate solo
informazioni disponibili prima della partita: ranking, punti, forma
recente, scontri diretti, rendimento sulla superficie, statistiche
storiche dei giocatori e, quando disponibili, le probabilita' implicite di
mercato.

La versione "senza quote" esclude `has_any_market_odds`,
`winner_implied_prob_avg`, `loser_implied_prob_avg` e
`implied_prob_diff_avg`, per isolare il contributo delle feature costruite
nella fase di feature engineering rispetto a quello delle quote dei
bookmaker.

## 2. Split cronologico: fold per la grid search e holdout finale

Una divisione casuale non e' appropriata: potrebbe addestrare il modello
su partite future e valutarlo su partite precedenti. Il 15% piu' recente
delle date uniche e' quindi riservato come **holdout finale**, mai
utilizzato durante il tuning. I fold di cross-validation per la grid
search sono costruiti sulle date del solo training set, cosi' le due
orientazioni della stessa partita (stessa data) restano sempre nello
stesso fold.

In [ ]:
train_idx, holdout_idx = chronological_holdout_split(
    dataset_with_odds.dates, HOLDOUT_FRACTION
)

X_train, y_train = dataset_with_odds.features.iloc[train_idx], dataset_with_odds.target.iloc[train_idx]
X_holdout, y_holdout = dataset_with_odds.features.iloc[holdout_idx], dataset_with_odds.target.iloc[holdout_idx]
dates_train = dataset_with_odds.dates.iloc[train_idx].reset_index(drop=True)

X_train_no_odds = dataset_without_odds.features.iloc[train_idx]
X_holdout_no_odds = dataset_without_odds.features.iloc[holdout_idx]

cv_folds = make_date_based_folds(dates_train, CV_FOLDS)

print(f"Training: {len(X_train):,} righe")
print(f"Holdout finale: {len(X_holdout):,} righe (ultimo {HOLDOUT_FRACTION:.0%} delle date)")
print(f"Fold di cross-validation: {len(cv_folds)}")

## 3. Baseline ingenua

Il classificatore va confrontato con una baseline semplice, che predice
sempre la classe maggioritaria. Poiche' il dataset simmetrico e'
bilanciato, l'accuratezza attesa della baseline e' circa il 50%.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_accuracy = accuracy_score(y_holdout, baseline.predict(X_holdout))

print(f"Accuratezza della baseline ingenua: {baseline_accuracy:.4f}")

## 4. Pipeline e grid search — modello con le feature di mercato

In [ ]:
def build_pipeline(numeric_features, categorical_features):
    preprocess = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), list(numeric_features)),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                list(categorical_features),
            ),
        ],
        remainder="drop",
    )
    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", DecisionTreeClassifier(criterion="gini", random_state=RANDOM_SEED)),
        ]
    )


param_grid = {
    "model__max_depth": [4, 6, 8, 10, 12, None],
    "model__min_samples_leaf": [1, 5, 20, 50],
    "model__min_samples_split": [2, 20, 50],
    "model__ccp_alpha": [0.0, 0.0005, 0.001, 0.005],
}

pipeline_with_odds = build_pipeline(
    dataset_with_odds.numeric_features, dataset_with_odds.categorical_features
)

search_with_odds = GridSearchCV(
    pipeline_with_odds,
    param_grid=param_grid,
    cv=cv_folds,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)
search_with_odds.fit(X_train, y_train)

print("Migliori parametri (con quote):", search_with_odds.best_params_)
print(f"Miglior accuratezza CV (con quote): {search_with_odds.best_score_:.4f}")

## 5. Pipeline e grid search — modello senza le feature di mercato

In [ ]:
pipeline_no_odds = build_pipeline(
    dataset_without_odds.numeric_features, dataset_without_odds.categorical_features
)

search_no_odds = GridSearchCV(
    pipeline_no_odds,
    param_grid=param_grid,
    cv=cv_folds,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
)
search_no_odds.fit(X_train_no_odds, y_train)

print("Migliori parametri (senza quote):", search_no_odds.best_params_)
print(f"Miglior accuratezza CV (senza quote): {search_no_odds.best_score_:.4f}")

## 6. Valutazione sull'holdout: con quote vs senza quote

In [ ]:
best_with_odds = search_with_odds.best_estimator_
best_no_odds = search_no_odds.best_estimator_

holdout_pred_with_odds = best_with_odds.predict(X_holdout)
holdout_pred_no_odds = best_no_odds.predict(X_holdout_no_odds)

comparison = pd.DataFrame(
    {
        "Con quote": {
            "Accuratezza CV": search_with_odds.best_score_,
            "Accuratezza holdout": accuracy_score(y_holdout, holdout_pred_with_odds),
        },
        "Senza quote": {
            "Accuratezza CV": search_no_odds.best_score_,
            "Accuratezza holdout": accuracy_score(y_holdout, holdout_pred_no_odds),
        },
    }
)
display(comparison)

### Matrice di confusione (modello con quote)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_holdout,
    holdout_pred_with_odds,
    display_labels=["Vince il giocatore B", "Vince il giocatore A"],
    cmap="Blues",
)
plt.title("Matrice di confusione — Decision Tree (con quote), holdout")
plt.grid(False)
plt.show()

### Precisione, richiamo e F1-score (modello con quote)

In [ ]:
print(
    classification_report(
        y_holdout,
        holdout_pred_with_odds,
        target_names=["Vince il giocatore B", "Vince il giocatore A"],
        digits=4,
    )
)

## 7. Importanza delle feature

L'importanza delle feature indica quanto ciascuna variabile contribuisce
a ridurre l'impurita' Gini lungo l'albero. Aiuta a interpretare il
modello, ma non dimostra un rapporto causale.

In [ ]:
def top_feature_importances(fitted_pipeline, top_n=10):
    preprocess = fitted_pipeline.named_steps["preprocess"]
    tree = fitted_pipeline.named_steps["model"]
    feature_names = preprocess.get_feature_names_out()
    importances = pd.Series(tree.feature_importances_, index=feature_names, name="importance")
    return importances.sort_values(ascending=False).head(top_n)


importances_with_odds = top_feature_importances(best_with_odds)
display(importances_with_odds.to_frame())

importances_with_odds.sort_values().plot(
    kind="barh", figsize=(8, 4.5), color="#2a9d8f", title="Top 10 feature — con quote"
)
plt.tight_layout()
plt.show()

### Importanza delle feature — modello senza quote

In [ ]:
importances_no_odds = top_feature_importances(best_no_odds)
display(importances_no_odds.to_frame())

importances_no_odds.sort_values().plot(
    kind="barh", figsize=(8, 4.5), color="#e76f51", title="Top 10 feature — senza quote"
)
plt.tight_layout()
plt.show()

## 8. Struttura dell'albero (primi livelli, modello con quote)

In [ ]:
fitted_preprocess = best_with_odds.named_steps["preprocess"]
fitted_tree = best_with_odds.named_steps["model"]
feature_names = fitted_preprocess.get_feature_names_out().tolist()

print(f"Profondita' albero: {fitted_tree.get_depth()}")
print(f"Numero di foglie: {fitted_tree.get_n_leaves()}")
print()
print(export_text(fitted_tree, feature_names=feature_names, max_depth=3))

fig, ax = plt.subplots(figsize=(20, 10))
plot_tree(
    fitted_tree,
    feature_names=feature_names,
    class_names=["player_a_loses", "player_a_wins"],
    filled=True,
    max_depth=3,
    fontsize=8,
    ax=ax,
)
plt.show()

## 9. Analisi delle partite piu' facili e piu' difficili (modello con quote)

Le righe sono abbinate tramite `match_id_internal`, cosi' ogni partita
reale viene contata una sola volta per questa diagnostica; le metriche
principali sopra restano quelle calcolate per riga. La probabilita'
assegnata al vincitore reale e' mediata tra i due orientamenti. Le
previsioni piu' corrette assegnano la probabilita' maggiore al vincitore
reale; quelle piu' errate la minore.

In [ ]:
holdout_probability = best_with_odds.predict_proba(X_holdout)[:, 1]

raw = pd.read_csv(FEATURES_PATH)
raw["date"] = pd.to_datetime(raw["date"], errors="raise")
holdout_raw = raw.iloc[holdout_idx].copy()

holdout_raw["actual_winner_probability"] = np.where(
    y_holdout.eq(1).to_numpy(), holdout_probability, 1 - holdout_probability
)
holdout_raw["actual_winner_probability"] = holdout_raw.groupby("match_id_internal")[
    "actual_winner_probability"
].transform("mean")

match_diagnostics = (
    holdout_raw.loc[holdout_raw["player_a_win"].eq(1)]
    .set_index("match_id_internal")
    .assign(correct=lambda df: df["actual_winner_probability"].ge(0.5))
)

instance_columns = [
    "date",
    "winner_name",
    "loser_name",
    "surface",
    "round",
    "winner_rank",
    "loser_rank",
    "actual_winner_probability",
]
instance_columns = [c for c in instance_columns if c in match_diagnostics.columns]

print("Partite nell'holdout:", len(match_diagnostics))
print(f"Accuratezza per partita: {match_diagnostics['correct'].mean():.4f}")

### Previsioni corrette con maggiore confidenza

In [ ]:
display(match_diagnostics.nlargest(10, "actual_winner_probability")[instance_columns])

### Previsioni errate con maggiore confidenza

In [ ]:
display(match_diagnostics.nsmallest(10, "actual_winner_probability")[instance_columns])

## Conclusioni

- La baseline ingenua ottiene circa il 50% di accuratezza, come atteso su
  un target bilanciato.
- Il confronto tra il modello con e senza le feature di mercato mostra
  quanto segnale predittivo autonomo portino ranking, punti e forma sulla
  superficie rispetto a quanto gia' incorporato nelle quote dei bookmaker.
- L'importanza delle feature e la struttura dell'albero permettono di
  verificare se il modello con le quote si riduce a un'unica soglia sulla
  probabilita' implicita, o se altre variabili contribuiscono in modo
  rilevante.
- L'analisi delle partite piu' difficili aiuta a capire se gli errori sono
  concentrati sui risultati a sorpresa (favorito sconfitto), coerentemente
  con quanto osservato per CatBoost.

Questo e' un rerun su un holdout esistente, non un nuovo test
indipendente: l'eventuale ulteriore tuning di feature o iperparametri va
fatto solo sui fold di cross-validation del training, non su questo
risultato.